In [2]:
import pandas as pd 
import numpy as numpy

In [14]:
df1 = pd.read_csv('master_question_practice_dataframe.csv')
df2 = pd.read_csv('numpy_pandas_practice_dataframe (1).csv')

df = df2.copy()

/tmp/ipykernel_2726/2536171668.py:1: DtypeWarning: Columns (0: scenario_id, 1: strategy, 2: asset, 3: asset_class, 4: sector, 5: region, 6: venue, 7: date, 8: timestamp, 9: batch_id, 10: order_id, 11: sequence_id, 12: stream_id, 13: matrix_id, 14: node_id, 15: neighbor_id, 16: interval_id, 17: string_group_id, 18: candidate_profile_id, 19: side, 20: exposure_style, 21: position_style, 22: job_status, 23: order_status, 24: alert_level, 25: exception_type, 26: duplicate_group, 27: token, 28: normalized_token, 29: string_value, 30: group_key, 31: cell_state) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv('master_question_practice_dataframe.csv')


In [15]:
market = df[df['record_type'] == 'market_bar'].copy()
orders = df[df['record_type'] == 'order_event'].copy()
borrow = df[df['record_type'] == 'borrow_snapshot'].copy()
surface = df[df['record_type'] == 'options_surface'].copy()
question_cols = [c for c in df.columns if c.startswith('q_')]
coverage = (
df[question_cols]
.sum()
.sort_values(ascending=False)
.rename_axis('question_flag')
.reset_index(name='support_rows')
)

5.1 Cross-sectional residual signal ranking
The hard part is not ranking. The hard part is deciding what should be ranked. Raw signal_z mixes sector effects and
factor exposure. A more honest score winsorizes by date, demeans within sector, and then residualizes against market,
size, and momentum betas.

In [ ]:
import numpy as np
import pandas as pd 

sub = df[df['record_type']=='market_bar'].copy()
sub = df[df['record_type'] == 'market_bar'].copy()

sub['wins_signal'] = sub.groupby('date')['signal_z'].transform(
    lambda s: s.clip(s.quantile(0.05), s.quantile(0.95))
)
sub['sector_neutral'] = sub.groupby(['date', 'sector'])['wins_signal'].transform(
    lambda s: s - s.mean()
)

def residualize_one_day(g):
    X = g[['factor_beta_mkt', 'factor_beta_size', 'factor_beta_mom']].to_numpy()
    X = np.c_[np.ones(len(g)), X]
    y = g['sector_neutral'].to_numpy()
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return pd.Series(y - X @ beta, index=g.index)

sub['resid_signal'] = (
    sub.groupby('date', group_keys=False)[
        ['sector_neutral', 'factor_beta_mkt', 'factor_beta_size', 'factor_beta_mom']
    ]
    .apply(lambda g: residualize_one_day(
        g.assign(sector_neutral=sub.loc[g.index, 'sector_neutral'])
    ))
)

sub['rank_pct'] = sub.groupby('date')['resid_signal'].rank(pct=True)

5.2 Borrow as-of alignment and squeeze detection
Borrow data lives on a slower clock than daily returns. That means the first nontrivial step is temporal alignment, not
aggregation. merge_asof is the correct primitive because each daily bar should inherit the latest known borrow state for
that asset.

In [37]:
market = df[df['record_type'] == 'market_bar'].copy()
borrow = df[df['record_type'] == 'borrow_snapshot'].copy()
market['date'] = pd.to_datetime(market['date'])
borrow['date'] = pd.to_datetime(borrow['date'])
parts = []

for asset, g in market.groupby('asset'):
    b = borrow[borrow['asset'] == asset][
    ['date', 'borrow_fee_bps', 'locate_available', 'utilization']
    ].sort_values('date')
    m = g.sort_values('date')
    if b.empty:
        m[['borrow_fee_latest', 'locate_latest', 'utilization_latest']] = np.nan
        parts.append(m)
    else:
        j = pd.merge_asof(m, b, on='date', direction='backward', suffixes=('', '_b'))
        j = j.rename(columns={
        'borrow_fee_bps_b': 'borrow_fee_latest',
        'locate_available_b': 'locate_latest',
        'utilization_b': 'utilization_latest'
        })
        parts.append(j)

aligned = pd.concat(parts, ignore_index=True).sort_values(['asset', 'date'])
aligned['fwd5_ret_bps'] = aligned.groupby('asset')['realized_return_bps'].transform(
    lambda s: s.shift(-1).rolling(5).sum()
)
aligned['squeeze_flag'] = (
    aligned['borrow_fee_latest'] >
    aligned.groupby('asset')['borrow_fee_latest'].transform(lambda s: s.quantile(0.90))
) & (
    aligned['utilization_latest'] >
    aligned.groupby('asset')['utilization_latest'].transform(lambda s: s.quantile(0.80))
)


In [41]:
orders = df[df['record_type']=='order_event'].copy()
sign = np.where(orders['side'].eq('BUY'), 1, -1)

orders['realized_shortfall_bps'] = (
    (orders['fill_price'] - orders['arrival_mid'])
    / orders['arrival_mid'] * 10000 * sign
)

orders['queue_loss_bps'] = np.log1p(orders['queue_ahead'])*0.38
orders['latency_cost_bps'] = 0.03 * orders['latency_ms']
orders['impact_component_bps'] = (
    orders['impact_bps']*(0.65+0.9*orders['participation_rate'])
)

orders['spread_component_bps'] = orders['spread_bps'] * 0.55

venue_rollup = (
    orders.groupby('venue')[
        ['realized_shortfall_bps', 'queue_loss_bps', 'latency_cost_bps',
         'impact_component_bps', 'spread_component_bps']
    ]
    .mean()
    .sort_values('realized_shortfall_bps', ascending=False)
)

In [42]:
market =  df[df['record_type']=='market_bar'].copy()
decile_perf = (
    market.dropna(subset=['signal_decile', 'next_return_bps'])
    .groupby('signal_decile')
    .agg(mean_next_ret_bps = ('next_return_bps', 'mean'),
         vol_next_ret_bps = ('next_return_bps', 'std'),
         avg_turnover = ('turnover', 'mean'),
         n = ('asset', 'count'))
)
decile_perf['ir_like'] = (
    decile_perf['mean_next_ret_bps'] / decile_perf['vol_next_ret_bps']
)

In [46]:
ev = df[
         (df['record_type']=='market_bar') &
         (df['event_id'].notna()) &
         (df['event_id'].astype(str) != '')
].copy()

profile = (
    ev.groupby(['event_type', 'days_from_event'])
    .agg(
        mean_abret_bps = ('abnormal_return_bps', 'mean'),
        n = ('asset', 'size'))
    .reset_index()
)
window = profile[profile['days_from_event'].between(-1,1)]
window

,event_type,days_from_event,mean_abret_bps,n
4,earnings,-1.0,-11.593039,4
5,earnings,0.0,151.758123,6
6,earnings,1.0,37.154440,6
15,guidance,-1.0,1.950346,4
16,guidance,0.0,80.184280,4
17,guidance,1.0,29.936415,4
26,product_launch,-1.0,-1.559348,3
27,product_launch,0.0,73.258673,5
28,product_launch,1.0,19.871247,3
36,regulatory,0.0,-68.372527,2


In [61]:
surf = df[df['record_type']=='options_surface'].copy()
surf['date'] = pd.to_datetime(surf['date'])
nvda_latest = surf[
    (surf['asset']=='NVDA') &
    (surf['date'] == surf['date'].max())
]
surface = nvda_latest.pivot_table(
    index = 'tenor_days',
    columns = 'delta_bucket',
    values = 'implied_vol'
).sort_index()
latest_date = nvda_latest['date'].max()
surface['put_call_skew']=surface['25P']-surface['25C']
surface['atm_term_change']=surface['50A'].diff()
iv_cols = ['25C', '50A', '25P']

In [62]:
import nbformat
import plotly.io._renderers
plotly.io._renderers.nbformat = nbformat

fig = go.Figure()

for tenor in surface.index:
    fig.add_trace(
        go.Scatter(
            x=iv_cols,
            y=surface.loc[tenor, iv_cols],
            mode="lines+markers",
            name=f"{tenor}D"
        )
    )

fig.update_layout(
    title=f"NVDA Volatility Smile by Tenor | {latest_date.date()}",
    xaxis_title="Delta Bucket",
    yaxis_title="Implied Volatility",
    legend_title="Tenor",
    template="plotly_white"
)

fig.show()

In [63]:
fig = go.Figure()

for delta_bucket in iv_cols:
    fig.add_trace(
        go.Scatter(
            x=surface.index,
            y=surface[delta_bucket],
            mode="lines+markers",
            name=delta_bucket
        )
    )

fig.update_layout(
    title=f"NVDA Implied Volatility Term Structure | {latest_date.date()}",
    xaxis_title="Tenor Days",
    yaxis_title="Implied Volatility",
    legend_title="Delta Bucket",
    template="plotly_white"
)

fig.show()

In [64]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=surface.index,
        y=surface["put_call_skew"],
        mode="lines+markers",
        name="25P minus 25C"
    )
)

fig.add_hline(
    y=0,
    line_dash="dash"
)

fig.update_layout(
    title=f"NVDA 25 Delta Put Call Skew | {latest_date.date()}",
    xaxis_title="Tenor Days",
    yaxis_title="25P IV minus 25C IV",
    template="plotly_white"
)

fig.show()

In [65]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=surface.index,
        y=surface["atm_term_change"],
        name="ATM Term Change"
    )
)

fig.add_hline(
    y=0,
    line_dash="dash"
)

fig.update_layout(
    title=f"NVDA ATM Implied Vol Term Change | {latest_date.date()}",
    xaxis_title="Tenor Days",
    yaxis_title="Change in 50A Implied Vol",
    template="plotly_white"
)

fig.show()

In [66]:
# Convert delta buckets to numeric x-axis locations.
delta_map = {
    "25P": -25,
    "50A": 50,
    "25C": 25
}

# Keep only columns that exist in the surface.
available_iv_cols = [col for col in iv_cols if col in surface.columns]

# X axis: mapped delta values.
x = [delta_map[col] for col in available_iv_cols]

# Y axis: tenor values.
y = surface.index.values

# Z axis: implied vol matrix.
z = surface[available_iv_cols].values

fig = go.Figure(
    data=[
        go.Surface(
            x=x,
            y=y,
            z=z,
            colorbar=dict(title="Implied Vol")
        )
    ]
)

fig.update_layout(
    title=f"NVDA 3D Implied Volatility Surface | {latest_date.date()}",
    scene=dict(
        xaxis_title="Delta Bucket",
        yaxis_title="Tenor Days",
        zaxis_title="Implied Volatility"
    ),
    template="plotly_white"
)

fig.show()

In [69]:
sub = df[df['record_type']=='interval_case'].copy()

def merge_one_asset(g):
    arr = g[['interval_start', 'interval_end']].sort_values('interval_start').to_numpy()
    merged = []
    for start, end in arr:
        if not merged or start > merged[-1][-1]:
            merged.append([start, end])
        else:
            merged[-1][1] = max(merged[-1][1], end)
    covered_days = sum(end-start + 1 for start, end in merged)
    return pd.Series({
        'raw_intervals':len(arr),
        'merged_interval':len(merged),
        'covered_days':covered_days
    })

intervals_summary = sub.groupby('asset').apply(merge_one_asset)
intervals_summary

,raw_intervals,merged_interval,covered_days
asset,,,
AAPL,12.0,7.0,145.0
AMZN,12.0,8.0,121.0
CL1,12.0,8.0,123.0
ES1,12.0,6.0,114.0
EURUSD,12.0,7.0,117.0
GC1,12.0,7.0,115.0
META,12.0,6.0,96.0
MSFT,12.0,7.0,149.0
NQ1,12.0,7.0,126.0


In [71]:
logs = df[df['record_type']=='automation_log']

logs['dup_key']=np.where(
    logs['duplicate_group'].eq(''),
    logs['row_id'],
    logs['duplicate_group']
)

dedup = (
    logs.groupby('dup_key')
    .agg(events=('row_id', 'size'),
         max_retry=('retry_count', 'max'),
         any_error=('exception_flag','max'),
         total_runtime=('run_time_sec','sum'))
)

job_rollup = (
    logs.groupby('job_name')
    .agg(total_runs=('row_id','size'),
         error_rate=('exception_flag', 'mean'),
         avg_retry=('retry_count','mean'),
         stale_rate=('stale_input_flag', 'mean'),
         avg_runtime_sec=('run_time_sec','mean'))
    .sort_values('error_rate', ascending=False)
)

In [ ]:
tags = df[df['record_type']=='string_case'].copy()

mapping = (
    tags.groupby(['normalized_tag', 'raw_tag'])
    .size()
    .reset_index(name='count')
    .sort_values(['normalized_tag', 'count'], ascending=[True, False])
)

dominant_raw = (
    mapping.groupby('normalized_tag')
    .head(3)
)


,normalized_tag,raw_tag,count
0,cross_sectional_equity,Cross Sectional Equity,11
1,cross_sectional_equity,x-sec eq,11
2,execution_overlay,exec overlay,10
3,financials,Fin.,11
4,financials,Financials,11
5,financials,fin srvcs,11
6,macro_overlay,Macro Overlay,11
7,macro_overlay,macro-ovrly,11
8,stat_arb,Stat Arb,10
9,stat_arb,stat-arb,10


In [ ]:
graph = df[df['record_type']=='graph_edge'].copy()
nodes = sorted(set(graph['node_id']).union(set(graph['neighbor_id'])) - {''})
idx = {node: i for i, node in enumerate(nodes)}

A 